# Week 4 Class

1. Rewrite the merge sort code, adding assertions to check that the loop invariant is satisfied.
2. Write code to sum the value in an array. Find a loop invariant for your code, and use it to prove that your code correctly sums those values.

## Note on assertions

In case you are not familiar, an [assertion](https://en.wikipedia.org/wiki/Assertion_(software_development)) is a debugging technique that can be used in any programming language used to catch bugs that might not result in the code raising an error, but might lead to the code giving incorrect results. In Python, you use the ``assert <condition>`` statement to say that at this point in the program, the ``<condition>`` must be true otherwise something has gone wrong. Python will evaluate ``<condition>`` and if it evaluates to ``True``, do nothing, but if it evaluates to ``False`` it will raise an error. Assertions are only evaluated in debug mode (the default on Python), so that in production code they have no computational cost.

Here's two examples to demonstrate what this looks like in Python.

In [1]:
def exactly_half_of(x):
    assert x%2==0 # can only return exactly half if it's divisible by 2
    return x//2 # integer division

exactly_half_of(4) # runs fine, assertion is True

2

In [2]:
exactly_half_of(3) # raises an AssertionError because 3 is not divisible by 2

AssertionError: 

Why might you want to use this? Well, just imagine in the code above if you didn't have an assertion and you did something like this (contrived example):

In [3]:
def half_of(n):
    return n//2 # integer division

def crazy_sum(X):
    n = len(X)
    half_n = half_of(n)
    left_sum = sum(X[0:half_n])
    right_sum = sum(X[half_n:2*half_n])
    return left_sum+right_sum

print(crazy_sum([1,2,3]))

3


You can see here that the sum returns 3 when you probably intended it to return 6. Your code has a bug but you don't get an error. If you had done the same thing with an assertion, it would raise an error and so you'd discover that your code had a hidden assumption that n is even.

In [4]:
def half_of(n):
    assert n%2==0
    return n//2 # integer division

def crazy_sum(X):
    n = len(X)
    half_n = half_of(n)
    left_sum = sum(X[0:half_n])
    right_sum = sum(X[half_n:2*half_n])
    return left_sum+right_sum

print(crazy_sum([1,2,3]))

AssertionError: 

## Task 1: Mergesort with loop invariant assertions

合并阶段那个 `while` 循环的 **loop invariant**（在每次循环开始时都成立）：

1. `len(X_tmp) == i + j` — 已经从 `L` 拿了 `i` 个、从 `R` 拿了 `j` 个，全都放进了 `X_tmp`
2. `X_tmp` 本身是已排序的（非递减）
3. `X_tmp` 里的所有元素都 ≤ `L[i:]` 和 `R[j:]` 中剩下的元素（"已挑出的都不大于还没挑的"）
4. `i <= len(L)` 且 `j <= len(R)` — 指针没越界

如果这四条在循环结束时仍然成立，且循环结束意味着 `i == len(L)` 且 `j == len(R)`，那么 `X_tmp` 就是 `L` 和 `R` 合并后的有序结果。

In [9]:
def merge_sort(X):
    if len(X) <= 1:
        return X
    mid = len(X) // 2
    R = merge_sort(X[:mid])
    L = merge_sort(X[mid:])
    i = j = 0
    X_tmp = []
    while i < len(L) or j < len(R):
        # ---- Loop invariant assertions ----
        # (1) X_tmp 的长度 = 已从 L、R 中取出的元素数
        assert len(X_tmp) == i + j
        # (2) 指针没越界
        assert 0 <= i <= len(L) and 0 <= j <= len(R)
        # (3) X_tmp 本身已排序（非递减）
        assert all(X_tmp[k] <= X_tmp[k + 1] for k in range(len(X_tmp) - 1))
        # (4) X_tmp 中所有元素 <= L[i:] 和 R[j:] 中剩下的元素
        if X_tmp:
            if i < len(L):
                assert X_tmp[-1] <= L[i]
            if j < len(R):
                assert X_tmp[-1] <= R[j]
        # -----------------------------------

        if i == len(L):
            X_tmp.append(R[j])
            j += 1
        elif j == len(R):
            X_tmp.append(L[i])
            i += 1
        else:
            if L[i] < R[j]:
                X_tmp.append(L[i])
                i += 1
            else:
                X_tmp.append(R[j])
                j += 1

    # 循环结束后：两个指针都到末尾，X_tmp 就是合并后的有序数组
    assert i == len(L) and j == len(R)
    assert len(X_tmp) == len(L) + len(R)
    return X_tmp


print(merge_sort([1, 8, 7, 3, 7, 10]))
print(merge_sort([6, 5, 3, 1, 8, 7, 2, 4]))
print(merge_sort([]))
print(merge_sort([42]))

[1, 3, 7, 7, 8, 10]
[1, 2, 3, 4, 5, 6, 7, 8]
[]
[42]
